# Backtest Simulation

Event-driven backtesting using the `backtest` library.

This notebook:
1. Loads market data from parquet stores
2. Merges events from multiple exchanges in chronological order
3. Runs simulation with ML model for trading decisions
4. Analyzes and visualizes results

In [ ]:
import sys

sys.path.insert(0, "../src")

import logging

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)

In [ ]:
from qate.core.model import ExchangeName, Market, Symbol
from qate.util.dt_range import DtRange

from backtest import (
    BacktestDataConfig,
    BacktestDataLoader,
    EventMerger,
    FeatureConfig,
    PositionConfig,
    SimulatorConfig,
    SimulatorV2,
    VolatilityConfig,
)

## Configuration

In [ ]:
# Data configuration
PARQUET_ROOT = "/data/parquet"  # Update this path
MODEL_PATH = "../models/lgbm_model.pkl"  # Update this path

# Date range for backtesting
START_DATE = "2025-01-15"
END_DATE = "2025-01-16"

# Markets
market_binance = Market(ExchangeName.BINANCE, Symbol.XRP_USDT)
market_gmo = Market(ExchangeName.GMO, Symbol.XRP_JPY)

In [ ]:
# Data loader configuration
data_config = BacktestDataConfig(
    dt_range=DtRange(START_DATE, END_DATE),
    market_binance=market_binance,
    market_gmo=market_gmo,
    parquet_root=PARQUET_ROOT,
)

In [ ]:
# Feature configuration
feature_config = FeatureConfig(
    book_levels=5,
    aggression_windows_sec=[1, 3, 5],
    book_change_depths=[5, 10, 20],
    depth_imbalance_rates=[0.001, 0.005, 0.01],
    min_orderbooks=10,
)

# Volatility configuration
volatility_config = VolatilityConfig(
    return_shift_sec=30,
    volatility_lookback_sec=1800,
    ema_span_sec=600,
    min_vol_spread_ratio=1.2,
    sample_interval_sec=1.0,
)

# Position configuration
position_config = PositionConfig(
    barrier_multiplier=0.7,
    timeout_sec=30,
)

In [ ]:
# Feature columns (must match training)
# Update this list based on your trained model's features
# You can get available features with: feature_calc.get_feature_names()
FEATURE_COLUMNS = [
    "spread_bps",
    "microprice_vs_mid",
    "volume_imbalance_1",
    "volume_imbalance_3",
    "volume_imbalance_5",
    "weighted_imbalance",
    "depth_imbalance_1bps",
    "depth_imbalance_5bps",
    "depth_imbalance_10bps",
    "depth_ratio_1",
    "depth_ratio_3",
    "depth_ratio_5",
    "aggression_imbalance_1s",
    "aggression_imbalance_3s",
    "aggression_imbalance_5s",
    "trades_per_sec_1s",
    "trades_per_sec_3s",
    "trades_per_sec_5s",
    "pull_imbalance_5",
    "refill_imbalance_5",
]

In [ ]:
# Simulator configuration
sim_config = SimulatorConfig(
    feature_config=feature_config,
    volatility_config=volatility_config,
    position_config=position_config,
    model_path=MODEL_PATH,
    feature_columns=FEATURE_COLUMNS,
    long_prediction=2,  # Model output for Long
    short_prediction=0,  # Model output for Short
)

## Load Data

In [ ]:
loader = BacktestDataLoader(data_config)

print("Loading BINANCE order books...")
binance_obs = loader.load_binance_orderbooks()

print("Loading BINANCE trades...")
binance_trades = loader.load_binance_trades()

print("Loading GMO order books...")
gmo_obs = loader.load_gmo_orderbooks()

print("\nLoaded:")
print(f"  BINANCE order books: {len(binance_obs):,}")
print(f"  BINANCE trades: {len(binance_trades):,}")
print(f"  GMO order books: {len(gmo_obs):,}")

## Run Simulation

In [ ]:
# Merge events from all sources
print("Merging events...")
events = EventMerger.merge(binance_obs, binance_trades, gmo_obs)

# Create simulator
simulator = SimulatorV2(sim_config)

# Run simulation
print("Running simulation...")
result = simulator.run(events)

## Analyze Results

In [ ]:
# Print summary
result.print_summary()

In [ ]:
# Get summary as dict
summary = result.summary()
summary

In [ ]:
# View trades as DataFrame
trades_df = result.to_dataframe()
trades_df.head(20)

## Visualizations

In [ ]:
# Plot cumulative PnL
result.plot_pnl()

In [ ]:
# Plot trades by hour
result.plot_trades_by_hour()

In [ ]:
# Plot PnL distribution
result.plot_pnl_distribution()

## Additional Analysis

In [ ]:
# PnL by exit reason
import pandas as pd

pnl_by_reason = result.get_pnl_by_exit_reason()
pd.DataFrame(pnl_by_reason).T

In [ ]:
# PnL by direction
pnl_by_direction = result.get_pnl_by_direction()
pd.DataFrame(pnl_by_direction).T

In [ ]:
# Hold time analysis
if not trades_df.empty:
    print("Hold Time Statistics (seconds):")
    print(trades_df["hold_time_sec"].describe())

In [ ]:
# PnL by hour heatmap
if not trades_df.empty:
    import matplotlib.pyplot as plt

    trades_df["hour"] = trades_df["entry_time"].dt.hour
    trades_df["date"] = trades_df["entry_time"].dt.date

    pivot = trades_df.pivot_table(
        values="pnl", index="date", columns="hour", aggfunc="sum", fill_value=0
    )

    plt.figure(figsize=(14, 4))
    plt.imshow(pivot.values, aspect="auto", cmap="RdYlGn")
    plt.colorbar(label="PnL")
    plt.xlabel("Hour")
    plt.ylabel("Date")
    plt.title("PnL Heatmap by Date and Hour")
    plt.xticks(range(24), range(24))
    plt.yticks(range(len(pivot.index)), pivot.index)
    plt.tight_layout()
    plt.show()

## Parameter Sensitivity (Optional)

Test different parameter combinations.

In [ ]:
# Example: Test different barrier multipliers
# Uncomment to run

# barrier_multipliers = [0.5, 0.7, 1.0, 1.5]
# results_by_multiplier = {}

# for mult in barrier_multipliers:
#     print(f"\nTesting barrier_multiplier = {mult}")
#
#     pos_config = PositionConfig(barrier_multiplier=mult, timeout_sec=30)
#     config = SimulatorConfig(
#         feature_config=feature_config,
#         volatility_config=volatility_config,
#         position_config=pos_config,
#         model_path=MODEL_PATH,
#         feature_columns=FEATURE_COLUMNS,
#     )
#
#     sim = SimulatorV2(config)
#     events = EventMerger.merge(binance_obs, binance_trades, gmo_obs)
#     res = sim.run(events)
#
#     results_by_multiplier[mult] = res.summary()
#     print(f"  Trades: {res.trade_count}, PnL: {res.total_pnl:.6f}, WR: {res.win_rate:.2%}")

# # Compare results
# pd.DataFrame({
#     m: {
#         'trades': s['trade_count'],
#         'total_pnl': s['total_pnl'],
#         'win_rate': s['win_rate'],
#         'profit_factor': s['profit_factor'],
#     }
#     for m, s in results_by_multiplier.items()
# }).T